# BEAM: Basic Usage Example

This notebook demonstrates how to use the BEAM package to load a pre-trained model and generate TESS images conditioned on orbital parameters.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch

# Add parent directory to path to import beam package
sys.path.append('..')

from beam.models import ContextUnet, DDPM
from beam.utils.visualization import plot_samples, plot_generation_process

## Load a Pre-trained Model

First, we'll create the model architecture and load pre-trained weights.

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Model parameters
n_feat = 256  # Number of features in U-Net
n_T = 600     # Number of diffusion timesteps

# Create model
unet = ContextUnet(in_channels=1, in_dim=12, n_feat=n_feat)
model = DDPM(nn_model=unet, betas=(1e-4, 0.02), n_T=n_T, device=device)

# Load pre-trained weights (adjust path as needed)
model_path = "../model_outputs/TESS_diffusion/model_epoch100.pth"

if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location=device)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    print(f"Loaded model from {model_path}")
else:
    print(f"No model found at {model_path}, using untrained model")

## Generate Random Samples

Let's generate some samples using random conditioning vectors.

In [ ]:
# Set model to evaluation mode
model.eval()

# Generate with random conditioning
n_sample = 5
image_shape = (64, 64)
guide_w = 1.0  # Guidance scale (0 = no guidance)

with torch.no_grad():
    # Generate with guidance
    x_gen, x_gen_store = model.sample(
        n_sample=n_sample,
        size=(1, image_shape[0], image_shape[1]),
        device=device,
        guide_w=guide_w
    )

# Plot results
fig, axes = plt.subplots(1, n_sample, figsize=(15, 3))
for i in range(n_sample):
    axes[i].imshow(x_gen[i][0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(f"Sample {i+1}")
    axes[i].axis('off')
    
plt.suptitle(f"Generated Samples (Guidance Scale = {guide_w})")
plt.tight_layout()
plt.show()

## Generate Samples with Specific Parameters

Now, let's generate samples using specific orbital parameters.

In [ ]:
# Create specific orbital parameters
# These should be normalized values similar to what the model was trained on
params = torch.tensor([
    # Parameter set 1
    [0.3, 0.5, 0.1, 0.3, 0.7, 0.2, 0.4, 0.6, 0.8, 0.9, 0.5, 0.3],
    # Parameter set 2
    [0.7, 0.2, 0.6, 0.8, 0.3, 0.9, 0.1, 0.5, 0.4, 0.2, 0.6, 0.7]
], dtype=torch.float32).reshape(2, 1, 12).to(device)

# Number of samples per parameter set
n_per_param = 3

with torch.no_grad():
    # Generate conditioned on parameters
    x_gen, x_gen_store = model.sample_c(
        c_i=params,
        n_sample=n_per_param,
        size=(1, image_shape[0], image_shape[1]),
        device=device
    )

# Plot results
fig, axes = plt.subplots(len(params), n_per_param, figsize=(12, 4 * len(params)))
for i in range(len(params)):
    for j in range(n_per_param):
        idx = i * n_per_param + j
        axes[i, j].imshow(x_gen[idx][0].cpu().numpy(), cmap='gray', vmin=0, vmax=1)
        axes[i, j].set_title(f"Param {i+1}, Sample {j+1}")
        axes[i, j].axis('off')
    
plt.suptitle("Generated Samples with Specific Parameters")
plt.tight_layout()
plt.show()

## Visualize the Generation Process

Let's visualize how the diffusion model gradually transforms noise into an image.

In [ ]:
# Plot the generation process for one sample
fig = plot_generation_process(
    x_gen_store=x_gen_store,
    sample_idx=0,
    num_timesteps=8,
    title="Diffusion Generation Process"
)
plt.show()

## Experiment with Different Guidance Scales

Let's see how different guidance scales affect the generated images.

In [ ]:
# Generate with different guidance scales
guidance_scales = [0.0, 0.5, 1.0, 2.0, 5.0]
samples = []

with torch.no_grad():
    for guide_w in guidance_scales:
        x_gen, _ = model.sample(
            n_sample=1,
            size=(1, image_shape[0], image_shape[1]),
            device=device,
            guide_w=guide_w
        )
        samples.append(x_gen[0][0].cpu().numpy())

# Plot results
fig, axes = plt.subplots(1, len(guidance_scales), figsize=(15, 3))
for i, (guide_w, img) in enumerate(zip(guidance_scales, samples)):
    axes[i].imshow(img, cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(f"Scale = {guide_w}")
    axes[i].axis('off')
    
plt.suptitle("Effect of Different Guidance Scales")
plt.tight_layout()
plt.show()